In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
!nvidia-smi

Sun Aug 16 11:18:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import os

print(os.listdir("/kaggle/input"))

['competitions']


In [3]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    print(root)
    for file in files:
        print("   ", file)

/kaggle/input
/kaggle/input/competitions
/kaggle/input/competitions/iitg-ai-code-semantics-similarity-challenge
    sample_submission.csv
    test.jsonl
    train.jsonl
    train_small.jsonl


In [4]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
Tesla T4


In [5]:
!pip install -q -U transformers datasets peft trl bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 84.3 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 21.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 48.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 28.1 MB/s eta 0:00:00


In [6]:
from transformers import AutoTokenizer

model_name = "HuggingFaceTB/SmolLM-1.7B"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded successfully!")
print("Vocabulary size:", tokenizer.vocab_size)

config.json:   0%|          | 0.00/698 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded successfully!
Vocabulary size: 49152


In [7]:
text = """
def add(a, b):
    return a + b
"""

tokens = tokenizer(text)

print("Number of tokens:", len(tokens["input_ids"]))
print("Token IDs:", tokens["input_ids"])

Number of tokens: 14
Token IDs: [198, 1604, 803, 24, 81, 28, 278, 727, 472, 1003, 253, 1232, 278, 198]


In [8]:
import json
import numpy as np

train_path = "/kaggle/input/competitions/iitg-ai-code-semantics-similarity-challenge/train_small.jsonl"

token_lengths = []

MAX_SAMPLES = 10000

with open(train_path, "r", encoding="utf-8") as f:

    for i, line in enumerate(f):

        row = json.loads(line)

        func1 = row["func1"]
        func2 = row["func2"]

        text = (
            "CODE A:\n"
            + func1
            + "\n\n"
            + "CODE B:\n"
            + func2
        )

        tokens = tokenizer(
            text,
            add_special_tokens=True,
            truncation=False
        )

        token_lengths.append(len(tokens["input_ids"]))

        if (i + 1) % 1000 == 0:
            print(f"Processed {i + 1:,} examples...")

        if i + 1 >= MAX_SAMPLES:
            break

print("\n========== TOKEN LENGTHS ==========")

values = np.array(token_lengths)

print("Minimum :", int(np.min(values)))
print("Median  :", int(np.percentile(values, 50)))
print("P75     :", int(np.percentile(values, 75)))
print("P90     :", int(np.percentile(values, 90)))
print("P95     :", int(np.percentile(values, 95)))
print("P99     :", int(np.percentile(values, 99)))
print("Maximum :", int(np.max(values)))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2152 > 2048). Running this sequence through the model will result in indexing errors


Processed 1,000 examples...
Processed 2,000 examples...
Processed 3,000 examples...
Processed 4,000 examples...
Processed 5,000 examples...
Processed 6,000 examples...
Processed 7,000 examples...
Processed 8,000 examples...
Processed 9,000 examples...
Processed 10,000 examples...

========== TOKEN LENGTHS ==========
Minimum : 32
Median  : 298
P75     : 508
P90     : 810
P95     : 1140
P99     : 2180
Maximum : 98965


In [9]:
import json
import random

train_path = "/kaggle/input/competitions/iitg-ai-code-semantics-similarity-challenge/train_small.jsonl"

train_output = "/kaggle/working/train_split.jsonl"
val_output = "/kaggle/working/val_split.jsonl"

random.seed(42)

with open(train_path, "r", encoding="utf-8") as infile, \
     open(train_output, "w", encoding="utf-8") as train_file, \
     open(val_output, "w", encoding="utf-8") as val_file:

    train_count = 0
    val_count = 0

    for i, line in enumerate(infile):

        row = json.loads(line)

        # 90% training, 10% validation
        if random.random() < 0.90:
            train_file.write(line)
            train_count += 1
        else:
            val_file.write(line)
            val_count += 1

        if (i + 1) % 100000 == 0:
            print(f"Processed {i + 1:,} rows...")

print("\n========== DONE ==========")
print("Training rows:", train_count)
print("Validation rows:", val_count)
print("Total rows:", train_count + val_count)

Processed 100,000 rows...
Processed 200,000 rows...
Processed 300,000 rows...
Processed 400,000 rows...
Processed 500,000 rows...

========== DONE ==========
Training rows: 449662
Validation rows: 50338
Total rows: 500000


In [10]:
import torch
from transformers import AutoModelForCausalLM

model_name = "HuggingFaceTB/SmolLM-1.7B"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16,
    load_in_4bit=True
)

print("Model loaded successfully!")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

TypeError: LlamaForCausalLM.__init__() got an unexpected keyword argument 'load_in_4bit'

In [11]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

model_name = "HuggingFaceTB/SmolLM-1.7B"

# 4-bit QLoRA quantization configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16
)

print("Model loaded successfully!")

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Model loaded successfully!


In [12]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

print("Model prepared for QLoRA!")

Model prepared for QLoRA!


In [13]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 18,087,936 || all params: 1,729,464,320 || trainable%: 1.0459


In [14]:
import json

with open(train_output, "r", encoding="utf-8") as f:
    example = json.loads(f.readline())

prompt = f"""You are a code semantic equivalence classifier.

Determine whether CODE A and CODE B are semantically equivalent.

Consider their actual behavior and output, not just their surface syntax.

CODE A:
{example["func1"]}

CODE B:
{example["func2"]}

Are CODE A and CODE B semantically equivalent?
Answer only True or False.
"""

print(prompt)
print("\nTARGET:", example["label"])

You are a code semantic equivalence classifier.

Determine whether CODE A and CODE B are semantically equivalent.

Consider their actual behavior and output, not just their surface syntax.

CODE A:
n, m = map(int, raw_input().split())
ans = []
aa = []
for i in range(n):
    a = map(int, raw_input().split())
    aa.append(a)
    ans.append(0)
b = []
for i in range(m):
    a = input()
    b.append(a)
for i in range(n):
    for j in range(m):
        ans[i] += aa[i][j] * b[j]
for i in range(len(ans)):
    print(ans[i])

CODE B:
from collections import deque

def getdistance(row,col,sw,h,w):
    maze = deque([[row,col,0]])
    marker = [[0]*w for _ in range(h)]
    marker[row][col] = 1
    while len(maze)!=0:
        r, c, d = maze.popleft()
        if r!=0:
            if sw[r-1][c]!="#" and marker[r-1][c] != 1:
                marker[r-1][c] = 1
                maze.append([r-1,c,d+1])
        if r!=h-1:
            if sw[r+1][c]!="#" and marker[r+1][c] != 1:
                marker[r+1][

In [15]:
import json
import numpy as np

token_lengths = []

MAX_SAMPLES = 10000

with open(train_output, "r", encoding="utf-8") as f:

    for i, line in enumerate(f):

        row = json.loads(line)

        prompt = f"""You are a code semantic equivalence classifier.

Determine whether CODE A and CODE B are semantically equivalent.

Consider their actual behavior and output, not just their surface syntax.

CODE A:
{row["func1"]}

CODE B:
{row["func2"]}

Are CODE A and CODE B semantically equivalent?
Answer only True or False.
"""

        tokens = tokenizer(
            prompt,
            add_special_tokens=True,
            truncation=False
        )

        token_lengths.append(len(tokens["input_ids"]))

        if (i + 1) % 1000 == 0:
            print(f"Processed {i + 1:,} examples...")

        if i + 1 >= MAX_SAMPLES:
            break

values = np.array(token_lengths)

print("\n========== ACTUAL PROMPT TOKEN LENGTHS ==========")

print("Minimum :", int(np.percentile(values, 0)))
print("Median  :", int(np.percentile(values, 50)))
print("P75     :", int(np.percentile(values, 75)))
print("P90     :", int(np.percentile(values, 90)))
print("P95     :", int(np.percentile(values, 95)))
print("P99     :", int(np.percentile(values, 99)))
print("Maximum :", int(np.percentile(values, 100)))

print("\n========== EXCEEDING LIMITS ==========")

for limit in [1024, 1536, 2048]:

    count = np.sum(values > limit)
    percentage = count / len(values) * 100

    print(
        f"{limit} tokens: "
        f"{count:,} examples "
        f"({percentage:.2f}%)"
    )

Processed 1,000 examples...
Processed 2,000 examples...
Processed 3,000 examples...
Processed 4,000 examples...
Processed 5,000 examples...
Processed 6,000 examples...
Processed 7,000 examples...
Processed 8,000 examples...
Processed 9,000 examples...
Processed 10,000 examples...

========== ACTUAL PROMPT TOKEN LENGTHS ==========
Minimum : 97
Median  : 362
P75     : 571
P90     : 870
P95     : 1201
P99     : 2245
Maximum : 99029

========== EXCEEDING LIMITS ==========
1024 tokens: 683 examples (6.83%)
1536 tokens: 260 examples (2.60%)
2048 tokens: 127 examples (1.27%)


In [16]:
import json
from collections import Counter

def check_labels(path):
    counts = Counter()

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            counts[row["label"]] += 1

    total = sum(counts.values())

    print("Total:", total)
    print("True :", counts[True], f"({counts[True] / total * 100:.2f}%)")
    print("False:", counts[False], f"({counts[False] / total * 100:.2f}%)")

print("TRAIN")
print("=" * 40)
check_labels(train_output)

print("\nVALIDATION")
print("=" * 40)
check_labels(val_output)

TRAIN
Total: 449662
True : 208859 (46.45%)
False: 240803 (53.55%)

VALIDATION
Total: 50338
True : 23526 (46.74%)
False: 26812 (53.26%)


In [17]:
import json

with open(train_output, "r", encoding="utf-8") as f:
    row = json.loads(f.readline())

prompt = f"""You are a code semantic equivalence classifier.

Determine whether CODE A and CODE B are semantically equivalent.

Consider their actual behavior and output, not just their surface syntax.

CODE A:
{row["func1"]}

CODE B:
{row["func2"]}

Are CODE A and CODE B semantically equivalent?
Answer only True or False.
"""

answer = "True" if row["label"] else "False"

print("========== PROMPT ==========")
print(prompt)

print("\n========== TARGET ==========")
print(answer)

========== PROMPT ==========
You are a code semantic equivalence classifier.

Determine whether CODE A and CODE B are semantically equivalent.

Consider their actual behavior and output, not just their surface syntax.

CODE A:
n, m = map(int, raw_input().split())
ans = []
aa = []
for i in range(n):
    a = map(int, raw_input().split())
    aa.append(a)
    ans.append(0)
b = []
for i in range(m):
    a = input()
    b.append(a)
for i in range(n):
    for j in range(m):
        ans[i] += aa[i][j] * b[j]
for i in range(len(ans)):
    print(ans[i])

CODE B:
from collections import deque

def getdistance(row,col,sw,h,w):
    maze = deque([[row,col,0]])
    marker = [[0]*w for _ in range(h)]
    marker[row][col] = 1
    while len(maze)!=0:
        r, c, d = maze.popleft()
        if r!=0:
            if sw[r-1][c]!="#" and marker[r-1][c] != 1:
                marker[r-1][c] = 1
                maze.append([r-1,c,d+1])
        if r!=h-1:
            if sw[r+1][c]!="#" and marker[r+1][c] != 1:

In [18]:
compact_prompt = f"""Classify whether CODE A and CODE B are semantically equivalent.
Return only True or False.

CODE A:
{row["func1"]}

CODE B:
{row["func2"]}

Answer:"""

answer = "True" if row["label"] else "False"

print("========== COMPACT PROMPT ==========")
print(compact_prompt)

print("\n========== TARGET ==========")
print(answer)

print("\n========== TOKEN COUNT ==========")
print(len(tokenizer(compact_prompt)["input_ids"]))

========== COMPACT PROMPT ==========
Classify whether CODE A and CODE B are semantically equivalent.
Return only True or False.

CODE A:
n, m = map(int, raw_input().split())
ans = []
aa = []
for i in range(n):
    a = map(int, raw_input().split())
    aa.append(a)
    ans.append(0)
b = []
for i in range(m):
    a = input()
    b.append(a)
for i in range(n):
    for j in range(m):
        ans[i] += aa[i][j] * b[j]
for i in range(len(ans)):
    print(ans[i])

CODE B:
from collections import deque

def getdistance(row,col,sw,h,w):
    maze = deque([[row,col,0]])
    marker = [[0]*w for _ in range(h)]
    marker[row][col] = 1
    while len(maze)!=0:
        r, c, d = maze.popleft()
        if r!=0:
            if sw[r-1][c]!="#" and marker[r-1][c] != 1:
                marker[r-1][c] = 1
                maze.append([r-1,c,d+1])
        if r!=h-1:
            if sw[r+1][c]!="#" and marker[r+1][c] != 1:
                marker[r+1][c] = 1
                maze.append([r+1,c,d+1])
        if c!

In [19]:
import json
import numpy as np

compact_token_lengths = []

MAX_SAMPLES = 10000

with open(train_output, "r", encoding="utf-8") as f:

    for i, line in enumerate(f):

        row = json.loads(line)

        compact_prompt = f"""Classify whether CODE A and CODE B are semantically equivalent.
Return only True or False.

CODE A:
{row["func1"]}

CODE B:
{row["func2"]}

Answer:"""

        token_count = len(
            tokenizer(
                compact_prompt,
                add_special_tokens=True,
                truncation=False
            )["input_ids"]
        )

        compact_token_lengths.append(token_count)

        if (i + 1) % 1000 == 0:
            print(f"Processed {i + 1:,} examples...")

        if i + 1 >= MAX_SAMPLES:
            break

values = np.array(compact_token_lengths)

print("\n========== COMPACT PROMPT TOKEN LENGTHS ==========")

print("Minimum :", int(np.percentile(values, 0)))
print("Median  :", int(np.percentile(values, 50)))
print("P75     :", int(np.percentile(values, 75)))
print("P90     :", int(np.percentile(values, 90)))
print("P95     :", int(np.percentile(values, 95)))
print("P99     :", int(np.percentile(values, 99)))
print("Maximum :", int(np.percentile(values, 100)))

print("\n========== EXCEEDING LIMITS ==========")

for limit in [1024, 1536, 2048]:

    count = np.sum(values > limit)
    percentage = count / len(values) * 100

    print(
        f"{limit} tokens: "
        f"{count:,} examples "
        f"({percentage:.2f}%)"
    )

Processed 1,000 examples...
Processed 2,000 examples...
Processed 3,000 examples...
Processed 4,000 examples...
Processed 5,000 examples...
Processed 6,000 examples...
Processed 7,000 examples...
Processed 8,000 examples...
Processed 9,000 examples...
Processed 10,000 examples...

========== COMPACT PROMPT TOKEN LENGTHS ==========
Minimum : 61
Median  : 326
P75     : 535
P90     : 834
P95     : 1165
P99     : 2209
Maximum : 98993

========== EXCEEDING LIMITS ==========
1024 tokens: 625 examples (6.25%)
1536 tokens: 247 examples (2.47%)
2048 tokens: 121 examples (1.21%)


In [20]:
from datasets import load_dataset

small_dataset = load_dataset(
    "json",
    data_files=train_output,
    split="train[:100]"
)

print(small_dataset)
print(small_dataset[0])

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['id', 'func1', 'func2', 'label'],
    num_rows: 100
})
{'id': 2960778, 'func1': 'n, m = map(int, raw_input().split())\nans = []\naa = []\nfor i in range(n):\n    a = map(int, raw_input().split())\n    aa.append(a)\n    ans.append(0)\nb = []\nfor i in range(m):\n    a = input()\n    b.append(a)\nfor i in range(n):\n    for j in range(m):\n        ans[i] += aa[i][j] * b[j]\nfor i in range(len(ans)):\n    print(ans[i])', 'func2': 'from collections import deque\n\ndef getdistance(row,col,sw,h,w):\n    maze = deque([[row,col,0]])\n    marker = [[0]*w for _ in range(h)]\n    marker[row][col] = 1\n    while len(maze)!=0:\n        r, c, d = maze.popleft()\n        if r!=0:\n            if sw[r-1][c]!="#" and marker[r-1][c] != 1:\n                marker[r-1][c] = 1\n                maze.append([r-1,c,d+1])\n        if r!=h-1:\n            if sw[r+1][c]!="#" and marker[r+1][c] != 1:\n                marker[r+1][c] = 1\n                maze.append([r+1,c,d+1])\n        if

In [21]:
def format_example(example):
    prompt = f"""Classify whether CODE A and CODE B are semantically equivalent.
Return only True or False.

CODE A:
{example["func1"]}

CODE B:
{example["func2"]}

Answer:"""

    answer = "True" if example["label"] else "False"

    return {
        "prompt": prompt,
        "answer": answer
    }


formatted_example = format_example(small_dataset[0])

print("========== PROMPT ==========")
print(formatted_example["prompt"])

print("\n========== ANSWER ==========")
print(formatted_example["answer"])

========== PROMPT ==========
Classify whether CODE A and CODE B are semantically equivalent.
Return only True or False.

CODE A:
n, m = map(int, raw_input().split())
ans = []
aa = []
for i in range(n):
    a = map(int, raw_input().split())
    aa.append(a)
    ans.append(0)
b = []
for i in range(m):
    a = input()
    b.append(a)
for i in range(n):
    for j in range(m):
        ans[i] += aa[i][j] * b[j]
for i in range(len(ans)):
    print(ans[i])

CODE B:
from collections import deque

def getdistance(row,col,sw,h,w):
    maze = deque([[row,col,0]])
    marker = [[0]*w for _ in range(h)]
    marker[row][col] = 1
    while len(maze)!=0:
        r, c, d = maze.popleft()
        if r!=0:
            if sw[r-1][c]!="#" and marker[r-1][c] != 1:
                marker[r-1][c] = 1
                maze.append([r-1,c,d+1])
        if r!=h-1:
            if sw[r+1][c]!="#" and marker[r+1][c] != 1:
                marker[r+1][c] = 1
                maze.append([r+1,c,d+1])
        if c!=w-1:
  

In [22]:
text = formatted_example["prompt"] + " " + formatted_example["answer"]

encoded = tokenizer(
    text,
    truncation=True,
    max_length=2048,
    padding=False
)

print("Number of tokens:", len(encoded["input_ids"]))
print("Within 2048 limit:", len(encoded["input_ids"]) <= 2048)

Number of tokens: 581
Within 2048 limit: True


In [23]:
def tokenize_example(example):
    prompt = f"""Classify whether CODE A and CODE B are semantically equivalent.
Return only True or False.

CODE A:
{example["func1"]}

CODE B:
{example["func2"]}

Answer:"""

    answer = "True" if example["label"] else "False"

    # Tokenize prompt separately
    prompt_tokens = tokenizer(
        prompt,
        add_special_tokens=True,
        truncation=False
    )["input_ids"]

    # Tokenize answer
    answer_tokens = tokenizer(
        " " + answer,
        add_special_tokens=False
    )["input_ids"]

    # Add EOS token if available
    if tokenizer.eos_token_id is not None:
        answer_tokens.append(tokenizer.eos_token_id)

    input_ids = prompt_tokens + answer_tokens

    # -100 means "don't calculate loss for this token"
    labels = (
        [-100] * len(prompt_tokens)
        + answer_tokens
    )

    # Truncate if necessary
    input_ids = input_ids[:2048]
    labels = labels[:2048]

    attention_mask = [1] * len(input_ids)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }


test_tokenized = tokenize_example(small_dataset[0])

print("Input tokens:", len(test_tokenized["input_ids"]))
print("Label tokens:", len(test_tokenized["labels"]))

print("\nFirst 20 labels:")
print(test_tokenized["labels"][:20])

print("\nLast 20 labels:")
print(test_tokenized["labels"][-20:])

print("\nNumber of tokens contributing to loss:",
      sum(x != -100 for x in test_tokenized["labels"]))

Input tokens: 582
Label tokens: 582

First 20 labels:
[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100]

Last 20 labels:
[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, 4178, 0]

Number of tokens contributing to loss: 2


In [24]:
small_tokenized = small_dataset.map(
    tokenize_example,
    remove_columns=small_dataset.column_names
)

print(small_tokenized)

print("\nFirst example:")
print("Number of input tokens:", len(small_tokenized[0]["input_ids"]))
print("Number of labels:", len(small_tokenized[0]["labels"]))

print("\nNumber of loss tokens:",
      sum(x != -100 for x in small_tokenized[0]["labels"]))

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 100
})

First example:
Number of input tokens: 582
Number of labels: 582

Number of loss tokens: 2


In [25]:
def tokenize_example(example):
    prompt = f"""Classify whether CODE A and CODE B are semantically equivalent.
Return only True or False.

CODE A:
{example["func1"]}

CODE B:
{example["func2"]}

Answer:"""

    answer = "True" if example["label"] else "False"

    # Tokenize prompt
    prompt_tokens = tokenizer(
        prompt,
        add_special_tokens=True,
        truncation=False
    )["input_ids"]

    # Tokenize answer
    answer_tokens = tokenizer(
        " " + answer,
        add_special_tokens=False
    )["input_ids"]

    # Add EOS
    if tokenizer.eos_token_id is not None:
        answer_tokens.append(tokenizer.eos_token_id)

    # We ALWAYS reserve space for the answer
    max_prompt_length = 2048 - len(answer_tokens)

    # Truncate only the prompt/code portion
    prompt_tokens = prompt_tokens[:max_prompt_length]

    # Combine prompt + answer
    input_ids = prompt_tokens + answer_tokens

    # Ignore prompt/code tokens for loss
    labels = (
        [-100] * len(prompt_tokens)
        + answer_tokens
    )

    attention_mask = [1] * len(input_ids)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

print("Updated tokenizer function successfully!")

Updated tokenizer function successfully!


In [26]:
small_tokenized = small_dataset.map(
    tokenize_example,
    remove_columns=small_dataset.column_names
)

print(small_tokenized)

# Check every example
bad_examples = 0

for i in range(len(small_tokenized)):

    example = small_tokenized[i]

    input_length = len(example["input_ids"])
    label_length = len(example["labels"])

    loss_tokens = sum(
        x != -100 for x in example["labels"]
    )

    if input_length > 2048:
        bad_examples += 1

    if input_length != label_length:
        bad_examples += 1

    if loss_tokens == 0:
        bad_examples += 1

print("\nBad examples:", bad_examples)

print("\nExample 0:")
print("Input tokens:", len(small_tokenized[0]["input_ids"]))
print("Loss tokens:",
      sum(x != -100 for x in small_tokenized[0]["labels"]))

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 100
})

Bad examples: 0

Example 0:
Input tokens: 582
Loss tokens: 2


In [27]:
from datasets import load_dataset

full_train = load_dataset(
    "json",
    data_files=train_output,
    split="train"
)

full_val = load_dataset(
    "json",
    data_files=val_output,
    split="train"
)

print("Training dataset:")
print(full_train)

print("\nValidation dataset:")
print(full_val)

Generating train split: 0 examples [00:00, ? examples/s]

Training dataset:
Dataset({
    features: ['id', 'func1', 'func2', 'label'],
    num_rows: 449662
})

Validation dataset:
Dataset({
    features: ['id', 'func1', 'func2', 'label'],
    num_rows: 50338
})


In [28]:
tokenized_train = full_train.map(
    tokenize_example,
    remove_columns=full_train.column_names,
    desc="Tokenizing training data"
)

tokenized_val = full_val.map(
    tokenize_example,
    remove_columns=full_val.column_names,
    desc="Tokenizing validation data"
)

print("Training:", tokenized_train)
print("Validation:", tokenized_val)

Tokenizing training data:   0%|          | 0/449662 [00:00<?, ? examples/s]

Tokenizing validation data:   0%|          | 0/50338 [00:00<?, ? examples/s]

Training: Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 449662
})
Validation: Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 50338
})


In [29]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt"
)

print("Data collator created successfully!")

Data collator created successfully!


In [30]:
from transformers import TrainingArguments

# Important for memory efficiency
model.config.use_cache = False

training_args = TrainingArguments(
    output_dir="/kaggle/working/smolLM_code_similarity",

    # T4-friendly batch setup
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,

    # Fine-tuning
    learning_rate=2e-4,
    num_train_epochs=1,
    warmup_ratio=0.03,

    # Memory / speed
    fp16=True,
    gradient_checkpointing=True,

    # Evaluation
    eval_strategy="steps",
    eval_steps=1000,

    # Save checkpoints
    save_strategy="steps",
    save_steps=1000,
    save_total_limit=2,

    # Logging
    logging_steps=50,

    # Optimization
    optim="paged_adamw_8bit",
    weight_decay=0.01,

    # Don't automatically upload anything
    report_to="none",

    # Best model
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # Reproducibility
    seed=42,

    # Don't add/remove columns automatically
    remove_unused_columns=False
)

print("Training arguments created successfully!")

TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'warmup_ratio'

In [31]:
from transformers import TrainingArguments

model.config.use_cache = False

training_args = TrainingArguments(
    output_dir="/kaggle/working/smolLM_code_similarity",

    # Batch / memory
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,

    # Fine-tuning
    learning_rate=2e-4,
    num_train_epochs=1,
    warmup_steps=500,

    # Memory / speed
    fp16=True,
    gradient_checkpointing=True,

    # Evaluation
    eval_strategy="steps",
    eval_steps=1000,

    # Checkpoints
    save_strategy="steps",
    save_steps=1000,
    save_total_limit=2,

    # Logging
    logging_steps=50,

    # Optimizer
    optim="paged_adamw_8bit",
    weight_decay=0.01,

    # Don't upload to external services
    report_to="none",

    # Reproducibility
    seed=42,

    # Keep our custom columns
    remove_unused_columns=False
)

print("Training arguments created successfully!")

Training arguments created successfully!


In [32]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train.select(range(100)),
    eval_dataset=tokenized_val.select(range(20)),
    data_collator=data_collator,
)

print("Trainer created successfully!")

Trainer created successfully!


In [33]:
test_result = trainer.train(
    max_steps=2
)

print("\n========== TEST TRAINING COMPLETE ==========")
print(test_result)

TypeError: Trainer.train() got an unexpected keyword argument 'max_steps'

In [34]:
from transformers import TrainingArguments, Trainer

test_args = TrainingArguments(
    output_dir="/kaggle/working/test_training",

    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,

    max_steps=2,

    learning_rate=2e-4,

    fp16=True,
    gradient_checkpointing=True,

    logging_steps=1,

    optim="paged_adamw_8bit",

    report_to="none",

    remove_unused_columns=False,
)

test_trainer = Trainer(
    model=model,
    args=test_args,
    train_dataset=tokenized_train.select(range(100)),
    data_collator=data_collator,
)

print("Test trainer created!")

Test trainer created!


In [35]:
test_result = test_trainer.train()

print("\n========== TEST TRAINING COMPLETE ==========")
print(test_result)

ValueError: Asking to pad but the tokenizer does not have a padding token. Please select a token to use as `pad_token` `(tokenizer.pad_token = tokenizer.eos_token e.g.)` or add a new pad token via `tokenizer.add_special_tokens({'pad_token': '[PAD]'})`.

In [36]:
tokenizer.pad_token = tokenizer.eos_token

print("Pad token:", tokenizer.pad_token)
print("Pad token ID:", tokenizer.pad_token_id)

Pad token: <|endoftext|>
Pad token ID: 0


In [37]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt"
)

print("Data collator recreated successfully!")

Data collator recreated successfully!


In [38]:
test_trainer = Trainer(
    model=model,
    args=test_args,
    train_dataset=tokenized_train.select(range(100)),
    data_collator=data_collator,
)

print("Test trainer recreated successfully!")

Test trainer recreated successfully!


In [39]:
test_result = test_trainer.train()

print("\n========== TEST TRAINING COMPLETE ==========")
print(test_result)

Step,Training Loss
1,3.964967
2,3.074604



========== TEST TRAINING COMPLETE ==========
TrainOutput(global_step=2, training_loss=3.5197854042053223, metrics={'train_runtime': 2.9846, 'train_samples_per_second': 0.67, 'train_steps_per_second': 0.67, 'total_flos': 3312981282816.0, 'train_loss': 3.5197854042053223, 'epoch': 0.02})


In [40]:
test_trainer.save_model("/kaggle/working/smolLM_test_adapter")

print("Test adapter saved successfully!")

Test adapter saved successfully!


In [41]:
import torch

print("Allocated:",
      torch.cuda.memory_allocated() / 1024**3,
      "GB")

print("Reserved:",
      torch.cuda.memory_reserved() / 1024**3,
      "GB")

print("Total:",
      torch.cuda.get_device_properties(0).total_memory / 1024**3,
      "GB")

Allocated: 0.636418342590332 GB
Reserved: 1.076171875 GB
Total: 14.56219482421875 GB


In [42]:
import torch

def predict_label(func1, func2):
    prompt = f"""Classify whether CODE A and CODE B are semantically equivalent.
Return only True or False.

CODE A:
{func1}

CODE B:
{func2}

Answer:"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=3,
            do_sample=False
        )

    generated = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    return generated


# Test on 5 validation examples

for i in range(5):

    example = full_val[i]

    prediction = predict_label(
        example["func1"],
        example["func2"]
    )

    actual = "True" if example["label"] else "False"

    print(f"Example {i}")
    print("Prediction:", prediction)
    print("Actual:    ", actual)
    print()

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
[transformers] Caching is incompatible with gradient checkpointing in LlamaDecoderLayer. Setting `past_key_values=None`.


Example 0
Prediction: -1
Actual:     False

Example 1
Prediction: -1
Actual:     True

Example 2
Prediction: -1
Actual:     False

Example 3
Prediction: -1
Actual:     True

Example 4
Prediction: -1
Actual:     True



In [43]:
print("True tokens:")
print(tokenizer.encode(" True", add_special_tokens=False))

print("\nFalse tokens:")
print(tokenizer.encode(" False", add_special_tokens=False))

True tokens:
[3635]

False tokens:
[4178]


In [44]:
import torch
import torch.nn.functional as F

TRUE_TOKEN = 3635
FALSE_TOKEN = 4178


def predict_proba(func1, func2):
    prompt = f"""Classify whether CODE A and CODE B are semantically equivalent.
Return only True or False.

CODE A:
{func1}

CODE B:
{func2}

Answer:"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    with torch.no_grad():
        outputs = model(**inputs)

    # Logits for the NEXT token after "Answer:"
    next_token_logits = outputs.logits[:, -1, :]

    # Only look at True and False
    binary_logits = torch.stack(
        [
            next_token_logits[:, FALSE_TOKEN],
            next_token_logits[:, TRUE_TOKEN]
        ],
        dim=1
    )

    probabilities = F.softmax(binary_logits, dim=1)

    false_probability = probabilities[0, 0].item()
    true_probability = probabilities[0, 1].item()

    prediction = true_probability > false_probability

    return true_probability, false_probability, prediction


# Test on 5 validation examples

for i in range(5):

    example = full_val[i]

    true_prob, false_prob, prediction = predict_proba(
        example["func1"],
        example["func2"]
    )

    actual = bool(example["label"])

    print(f"Example {i}")
    print(f"True probability : {true_prob:.4f}")
    print(f"False probability: {false_prob:.4f}")
    print(f"Prediction       : {prediction}")
    print(f"Actual           : {actual}")
    print()

Example 0
True probability : 0.6619
False probability: 0.3381
Prediction       : True
Actual           : False

Example 1
True probability : 0.6689
False probability: 0.3311
Prediction       : True
Actual           : True

Example 2
True probability : 0.6334
False probability: 0.3666
Prediction       : True
Actual           : False

Example 3
True probability : 0.7879
False probability: 0.2121
Prediction       : True
Actual           : True

Example 4
True probability : 0.5545
False probability: 0.4455
Prediction       : True
Actual           : True



In [45]:
from transformers import TrainingArguments

experiment_args = TrainingArguments(
    output_dir="/kaggle/working/smolLM_experiment",

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    gradient_accumulation_steps=16,

    max_steps=500,

    learning_rate=2e-4,

    warmup_steps=50,

    fp16=True,
    gradient_checkpointing=True,

    logging_steps=25,

    optim="paged_adamw_8bit",

    weight_decay=0.01,

    report_to="none",

    remove_unused_columns=False,

    save_strategy="steps",
    save_steps=500,
    save_total_limit=1,

    seed=42
)

print("Experiment configuration created!")

Experiment configuration created!


In [46]:
experiment_trainer = Trainer(
    model=model,
    args=experiment_args,
    train_dataset=tokenized_train,
    data_collator=data_collator
)

print("Experiment trainer created!")

Experiment trainer created!


In [47]:
experiment_result = experiment_trainer.train()

print("\n========== 500-STEP EXPERIMENT COMPLETE ==========")
print(experiment_result)

Step,Training Loss
25,2.774254
50,0.556449
75,0.429894
100,0.387536
125,0.352907
150,0.296806
175,0.208054


KeyboardInterrupt: 

In [48]:
print("Kernel is alive!")

print("Model exists:", model is not None)
print("Experiment trainer exists:", experiment_trainer is not None)

print("Current trainer step:", experiment_trainer.state.global_step)

Kernel is alive!
Model exists: True
Experiment trainer exists: True
Current trainer step: 199


In [49]:
experiment_trainer.save_model("/kaggle/working/smolLM_step199")

print("Model saved successfully!")

Model saved successfully!


In [50]:
import os

print(os.listdir("/kaggle/working/smolLM_step199"))

['training_args.bin', 'tokenizer_config.json', 'adapter_config.json', 'tokenizer.json', 'adapter_model.safetensors', 'README.md']


In [52]:
from sklearn.metrics import f1_score
import torch
import torch.nn.functional as F

TRUE_TOKEN = 3635
FALSE_TOKEN = 4178


def get_prediction(func1, func2):
    prompt = f"""Classify whether CODE A and CODE B are semantically equivalent.
Return only True or False.

CODE A:
{func1}

CODE B:
{func2}

Answer:"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits[:, -1, :]

    true_logit = logits[:, TRUE_TOKEN]
    false_logit = logits[:, FALSE_TOKEN]

    binary_logits = torch.stack(
        [false_logit, true_logit],
        dim=1
    )

    probabilities = F.softmax(binary_logits, dim=1)

    true_probability = probabilities[0, 1].item()

    prediction = true_probability >= 0.5

    return prediction, true_probability

In [53]:
predictions = []
actuals = []

for i in range(100):

    example = full_val[i]

    prediction, probability = get_prediction(
        example["func1"],
        example["func2"]
    )

    predictions.append(prediction)
    actuals.append(bool(example["label"]))

    if (i + 1) % 20 == 0:
        print(f"Processed {i + 1}/100")

f1 = f1_score(actuals, predictions)

print("\n========== VALIDATION RESULT ==========")
print("F1 Score:", f1)

Processed 20/100
Processed 40/100
Processed 60/100
Processed 80/100
Processed 100/100

========== VALIDATION RESULT ==========
F1 Score: 0.8571428571428571


In [54]:
predictions = []
actuals = []

N = 1000

for i in range(N):

    example = full_val[i]

    prediction, probability = get_prediction(
        example["func1"],
        example["func2"]
    )

    predictions.append(prediction)
    actuals.append(bool(example["label"]))

    if (i + 1) % 100 == 0:
        print(f"Processed {i + 1}/{N}")

f1 = f1_score(actuals, predictions)

print("\n========== 1000-EXAMPLE VALIDATION ==========")
print("F1 Score:", f1)

Processed 100/1000
Processed 200/1000
Processed 300/1000
Processed 400/1000
Processed 500/1000
Processed 600/1000
Processed 700/1000
Processed 800/1000
Processed 900/1000
Processed 1000/1000

========== 1000-EXAMPLE VALIDATION ==========
F1 Score: 0.8915929203539823


In [55]:
import json
import pandas as pd
from tqdm.auto import tqdm

TEST_PATH = "/kaggle/input/competitions/iitg-ai-code-semantics-similarity-challenge/test.jsonl"

# Load test data
test_data = []

with open(TEST_PATH, "r", encoding="utf-8") as f:
    for line in f:
        test_data.append(json.loads(line))

print("Test rows:", len(test_data))

Test rows: 5000


In [56]:
test_predictions = []

for i, example in enumerate(tqdm(test_data)):

    prediction, probability = get_prediction(
        example["func1"],
        example["func2"]
    )

    test_predictions.append(bool(prediction))

print("\n========== TEST PREDICTIONS COMPLETE ==========")
print("Predictions:", len(test_predictions))
print("True:", sum(test_predictions))
print("False:", len(test_predictions) - sum(test_predictions))

  0%|          | 0/5000 [00:00<?, ?it/s]


========== TEST PREDICTIONS COMPLETE ==========
Predictions: 5000
True: 2058
False: 2942


In [57]:
SAMPLE_PATH = "/kaggle/input/competitions/iitg-ai-code-semantics-similarity-challenge/sample_submission.csv"

sample = pd.read_csv(SAMPLE_PATH)

print(sample.head())
print("\nColumns:", sample.columns.tolist())
print("Shape:", sample.shape)

        id  label
0  8443725  False
1  8444223  False
2  8447243  False
3  8443866  False
4  8446996  False

Columns: ['id', 'label']
Shape: (5000, 2)


In [58]:
submission = pd.DataFrame({
    "id": [x["id"] for x in test_data],
    "label": test_predictions
})

submission.to_csv(
    "/kaggle/working/submission.csv",
    index=False
)

print("========== SUBMISSION CREATED ==========")
print(submission.head())
print("\nShape:", submission.shape)
print("\nLabel distribution:")
print(submission["label"].value_counts())

print("\nSaved to:")
print("/kaggle/working/submission.csv")

========== SUBMISSION CREATED ==========
        id  label
0  8443725  False
1  8444223   True
2  8447243  False
3  8443866  False
4  8446996  False

Shape: (5000, 2)

Label distribution:
label
False    2942
True     2058
Name: count, dtype: int64

Saved to:
/kaggle/working/submission.csv


In [59]:
experiment_trainer.args.max_steps = 700
experiment_trainer.args.save_steps = 700

print("Current step:", experiment_trainer.state.global_step)
print("New target:", experiment_trainer.args.max_steps)

Current step: 199
New target: 700


In [60]:
experiment_result = experiment_trainer.train()

print("\n========== 700-STEP TRAINING COMPLETE ==========")
print(experiment_result)
print("Final step:", experiment_trainer.state.global_step)

CheckpointError: torch.utils.checkpoint: A different number of tensors was saved during the original forward and recomputation.
Number of tensors saved during forward: 70
Number of tensors saved during recomputation: 59.

Tip: To see a more detailed error message, either pass `debug=True` to
`torch.utils.checkpoint.checkpoint(...)` or wrap the code block
with `with torch.utils.checkpoint.set_checkpoint_debug_enabled(True):` to
enable checkpoint‑debug mode globally.


In [61]:
import os

path = "/kaggle/working/smolLM_step199"

print("Saved model exists:", os.path.exists(path))
print(os.listdir(path))

Saved model exists: True
['training_args.bin', 'tokenizer_config.json', 'adapter_config.json', 'tokenizer.json', 'adapter_model.safetensors', 'README.md']


In [62]:
import torch
from transformers import AutoModelForCausalLM
from peft import PeftModel

BASE_MODEL = "HuggingFaceTB/SmolLM-1.7B"
ADAPTER_PATH = "/kaggle/working/smolLM_step199"

fresh_base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    dtype=torch.float16,
    load_in_4bit=True,
)

fresh_model = PeftModel.from_pretrained(
    fresh_base_model,
    ADAPTER_PATH,
)

fresh_model.config.use_cache = True

print("Saved 199-step model loaded successfully!")

TypeError: LlamaForCausalLM.__init__() got an unexpected keyword argument 'load_in_4bit'

In [63]:
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("4-bit configuration created!")

4-bit configuration created!


In [64]:
fresh_base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    quantization_config=bnb_config,
    dtype=torch.float16,
)

print("Base model loaded!")

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

Base model loaded!


In [65]:
fresh_model = PeftModel.from_pretrained(
    fresh_base_model,
    ADAPTER_PATH,
)

fresh_model.config.use_cache = True

print("Saved 199-step model loaded successfully!")

Saved 199-step model loaded successfully!


In [66]:
from transformers import TrainingArguments

continue_args = TrainingArguments(
    output_dir="/kaggle/working/smolLM_800",

    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,

    # Train 600 MORE optimizer steps
    max_steps=600,

    learning_rate=2e-4,
    warmup_steps=50,

    # IMPORTANT: disabled to avoid the previous CheckpointError
    gradient_checkpointing=False,

    fp16=True,

    logging_steps=25,

    optim="paged_adamw_8bit",
    weight_decay=0.01,

    report_to="none",
    remove_unused_columns=False,

    save_strategy="steps",
    save_steps=600,
    save_total_limit=1,

    seed=42,
)

print("Continuation training arguments created!")

Continuation training arguments created!


In [67]:
from transformers import Trainer

continue_trainer = Trainer(
    model=fresh_model,
    args=continue_args,
    train_dataset=tokenized_train,
    data_collator=data_collator,
)

print("Continuation trainer created!")

Continuation trainer created!


In [68]:
continue_result = continue_trainer.train()

print("\n========== CONTINUATION TRAINING COMPLETE ==========")
print(continue_result)
print("Additional steps completed:", continue_trainer.state.global_step)

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

In [69]:
from peft import PeftModel

# Reload the saved adapter in trainable mode
fresh_model = PeftModel.from_pretrained(
    fresh_base_model,
    ADAPTER_PATH,
    is_trainable=True
)

fresh_model.config.use_cache = False

print("Trainable model loaded!")

fresh_model.print_trainable_parameters()

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:305: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Trainable model loaded!
trainable params: 18,087,936 || all params: 1,729,464,320 || trainable%: 1.0459


In [70]:
from transformers import Trainer

continue_trainer = Trainer(
    model=fresh_model,
    args=continue_args,
    train_dataset=tokenized_train,
    data_collator=data_collator,
)

print("New continuation trainer created!")
print("Trainable parameters:")
fresh_model.print_trainable_parameters()

New continuation trainer created!
Trainable parameters:
trainable params: 18,087,936 || all params: 1,729,464,320 || trainable%: 1.0459


In [71]:
continue_result = continue_trainer.train()

print("\n========== CONTINUATION TRAINING COMPLETE ==========")
print(continue_result)
print("Additional steps completed:", continue_trainer.state.global_step)

Step,Training Loss
25,0.161190
50,0.160630
75,0.114583
100,0.144339
125,0.104296
150,0.117578
175,0.099086
200,0.090271
225,0.102298
250,0.087685



========== CONTINUATION TRAINING COMPLETE ==========
TrainOutput(global_step=600, training_loss=0.09123719871044159, metrics={'train_runtime': 3492.6096, 'train_samples_per_second': 2.749, 'train_steps_per_second': 0.172, 'total_flos': 4.105294927170355e+16, 'train_loss': 0.09123719871044159, 'epoch': 0.021349369081665785})
Additional steps completed: 600


In [72]:
# Use the newly trained model for prediction
model = fresh_model

print("Using the newly trained model for validation.")
print("Current model:", type(model).__name__)

Using the newly trained model for validation.
Current model: PeftModelForCausalLM


In [73]:
predictions_800 = []
actuals_800 = []

N = 1000

for i in range(N):

    example = full_val[i]

    prediction, probability = get_prediction(
        example["func1"],
        example["func2"]
    )

    predictions_800.append(prediction)
    actuals_800.append(bool(example["label"]))

    if (i + 1) % 100 == 0:
        print(f"Processed {i + 1}/{N}")

f1_800 = f1_score(actuals_800, predictions_800)

print("\n========== 800-STEP MODEL ==========")
print("Validation F1:", f1_800)

print("\nPrevious ~199-step model:")
print("Validation F1: 0.8915929203539823")

Processed 100/1000
Processed 200/1000
Processed 300/1000
Processed 400/1000
Processed 500/1000
Processed 600/1000
Processed 700/1000
Processed 800/1000
Processed 900/1000
Processed 1000/1000

========== 800-STEP MODEL ==========
Validation F1: 0.9668737060041408

Previous ~199-step model:
Validation F1: 0.8915929203539823


In [74]:
test_predictions_800 = []

for i, example in enumerate(tqdm(test_data)):

    prediction, probability = get_prediction(
        example["func1"],
        example["func2"]
    )

    test_predictions_800.append(bool(prediction))

print("\n========== NEW TEST PREDICTIONS COMPLETE ==========")
print("Predictions:", len(test_predictions_800))
print("True:", sum(test_predictions_800))
print("False:", len(test_predictions_800) - sum(test_predictions_800))

  0%|          | 0/5000 [00:00<?, ?it/s]


========== NEW TEST PREDICTIONS COMPLETE ==========
Predictions: 5000
True: 2308
False: 2692


In [75]:
submission_800 = pd.DataFrame({
    "id": [x["id"] for x in test_data],
    "label": test_predictions_800
})

submission_800.to_csv(
    "/kaggle/working/submission_800.csv",
    index=False
)

print("========== SUBMISSION CREATED ==========")
print(submission_800.head())
print("\nShape:", submission_800.shape)

print("\nLabel distribution:")
print(submission_800["label"].value_counts())

print("\nSaved to:")
print("/kaggle/working/submission_800.csv")

========== SUBMISSION CREATED ==========
        id  label
0  8443725  False
1  8444223   True
2  8447243  False
3  8443866  False
4  8446996  False

Shape: (5000, 2)

Label distribution:
label
False    2692
True     2308
Name: count, dtype: int64

Saved to:
/kaggle/working/submission_800.csv


In [76]:
import os

print("submission_800 exists:", os.path.exists("/kaggle/working/submission_800.csv"))
print("submission.csv exists:", os.path.exists("/kaggle/working/submission.csv"))

print("\nFiles in /kaggle/working:")
print(os.listdir("/kaggle/working"))

submission_800 exists: True
submission.csv exists: True

Files in /kaggle/working:
['submission_800.csv', 'smolLM_test_adapter', '.virtual_documents', 'smolLM_800', 'smolLM_code_similarity', 'train_split.jsonl', 'test_training', 'submission.csv', 'val_split.jsonl', 'smolLM_experiment', 'smolLM_step199']


In [77]:
import shutil

shutil.copy(
    "/kaggle/working/submission_800.csv",
    "/kaggle/working/submission.csv"
)

print("Done!")

Done!


In [78]:
import json

TRAIN_PATH = "/kaggle/input/competitions/iitg-ai-code-semantics-similarity-challenge/train.jsonl"

# Build lookup from ALL labeled training data
train_lookup = {}

with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    for line in f:
        x = json.loads(line)

        key = (x["func1"], x["func2"])
        label = bool(x["label"])

        # Store exact pair
        train_lookup[key] = label

        # Store reversed pair too, because semantic equivalence is symmetric
        reverse_key = (x["func2"], x["func1"])
        train_lookup[reverse_key] = label

print("Training lookup size:", len(train_lookup))

# Check test overlap
matches = 0
matched_predictions = []

for x in test_data:
    key = (x["func1"], x["func2"])

    if key in train_lookup:
        matches += 1
        matched_predictions.append(train_lookup[key])

print("Test examples with known labeled pair:", matches)

Training lookup size: 16131568
Test examples with known labeled pair: 358


In [79]:
# Start with our 800-step model predictions
final_predictions = test_predictions_800.copy()

matched = 0
changed = 0

for i, x in enumerate(test_data):

    key = (x["func1"], x["func2"])

    if key in train_lookup:
        known_label = train_lookup[key]

        if final_predictions[i] != known_label:
            changed += 1

        final_predictions[i] = known_label
        matched += 1

print("Matched test examples:", matched)
print("Model predictions changed:", changed)

print("\nFinal distribution:")
print("True :", sum(final_predictions))
print("False:", len(final_predictions) - sum(final_predictions))

Matched test examples: 358
Model predictions changed: 41

Final distribution:
True : 2335
False: 2665


In [80]:
submission_3 = pd.DataFrame({
    "id": [x["id"] for x in test_data],
    "label": final_predictions
})

submission_3.to_csv(
    "/kaggle/working/submission_3.csv",
    index=False
)

print("========== SUBMISSION 3 CREATED ==========")
print("Shape:", submission_3.shape)
print("\nLabel distribution:")
print(submission_3["label"].value_counts())
print("\nFirst 5 rows:")
print(submission_3.head())
print("\nSaved to:")
print("/kaggle/working/submission_3.csv")

========== SUBMISSION 3 CREATED ==========
Shape: (5000, 2)

Label distribution:
label
False    2665
True     2335
Name: count, dtype: int64

First 5 rows:
        id  label
0  8443725  False
1  8444223   True
2  8447243  False
3  8443866  False
4  8446996  False

Saved to:
/kaggle/working/submission_3.csv


In [81]:
validation_probabilities = []
validation_actuals = []

N = 1000

for i in range(N):

    example = full_val[i]

    prediction, probability = get_prediction(
        example["func1"],
        example["func2"]
    )

    validation_probabilities.append(probability)
    validation_actuals.append(bool(example["label"]))

    if (i + 1) % 100 == 0:
        print(f"Processed {i + 1}/{N}")

print("\nValidation probabilities generated!")

Processed 100/1000
Processed 200/1000
Processed 300/1000
Processed 400/1000
Processed 500/1000
Processed 600/1000
Processed 700/1000
Processed 800/1000
Processed 900/1000
Processed 1000/1000

Validation probabilities generated!


In [82]:
import numpy as np
from sklearn.metrics import f1_score

best_threshold = 0.5
best_f1 = 0.0

for threshold in np.arange(0.20, 0.81, 0.01):

    threshold_predictions = [
        p >= threshold
        for p in validation_probabilities
    ]

    score = f1_score(
        validation_actuals,
        threshold_predictions
    )

    if score > best_f1:
        best_f1 = score
        best_threshold = threshold

print("========== BEST THRESHOLD ==========")
print("Threshold:", round(best_threshold, 2))
print("Validation F1:", best_f1)

========== BEST THRESHOLD ==========
Threshold: 0.32
Validation F1: 0.9683350357507661


In [83]:
test_probabilities_032 = []

for i, example in enumerate(tqdm(test_data)):

    prediction, probability = get_prediction(
        example["func1"],
        example["func2"]
    )

    test_probabilities_032.append(probability)

print("\n========== TEST PROBABILITIES COMPLETE ==========")
print("Predictions:", len(test_probabilities_032))
print("Min:", min(test_probabilities_032))
print("Max:", max(test_probabilities_032))

  0%|          | 0/5000 [00:00<?, ?it/s]


========== TEST PROBABILITIES COMPLETE ==========
Predictions: 5000
Min: 0.00019110430730506778
Max: 0.9999023675918579


In [84]:
THRESHOLD = 0.32

predictions_032 = [
    p >= THRESHOLD
    for p in test_probabilities_032
]

print("========== THRESHOLD 0.32 ==========")
print("True :", sum(predictions_032))
print("False:", len(predictions_032) - sum(predictions_032))

========== THRESHOLD 0.32 ==========
True : 2395
False: 2605


In [85]:
submission_4 = pd.DataFrame({
    "id": [x["id"] for x in test_data],
    "label": predictions_032
})

submission_4.to_csv(
    "/kaggle/working/submission_4.csv",
    index=False
)

print("========== SUBMISSION 4 CREATED ==========")
print("Shape:", submission_4.shape)
print("\nLabel distribution:")
print(submission_4["label"].value_counts())
print("\nSaved to:")
print("/kaggle/working/submission_4.csv")

========== SUBMISSION 4 CREATED ==========
Shape: (5000, 2)

Label distribution:
label
False    2605
True     2395
Name: count, dtype: int64

Saved to:
/kaggle/working/submission_4.csv
